In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

# Models:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.classification import MultilayerPerceptronClassifier

In [0]:

auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
recall_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="recallByLabel"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="fMeasureByLabel"
)


In [0]:

df = spark.read.table("teams.sensorx.df_sx_800_with_delta")
df = df.drop("serialNumber", "payload_fault_faultName", "GeneratorType")

sensor_cols = [c for c in df.columns if c not in {"properties_deviceId", "timeStamp", "payload_fault_state"}]

df = df.na.drop(subset=sensor_cols)

# --- 2) Encode properties_deviceId as integer index (pure SQL, no MLlib fit) ---
device_ids = df.select("properties_deviceId").distinct().orderBy("properties_deviceId")
device_ids = device_ids.withColumn("deviceId_index", (F.dense_rank().over(Window.orderBy("properties_deviceId")) - 1).cast("double"))
df_encoded = df.join(device_ids, on="properties_deviceId", how="inner")

# ====================================================================
# CONFIGURABLE HORIZONS (adjust these to experiment)
# Each value defines a class boundary in days.
# Classes are assigned from most urgent to least:
#   Class 3: failure within 5 days  (imminent)
#   Class 2: failure within 10 days (but not 5)
#   Class 1: failure within 15 days (but not 10)
#   Class 0: no failure within 15 days (safe)
# ====================================================================
horizon_days = [5, 10, 15]  # <-- Change these to experiment!

# Compute failure flag for each horizon window
for h in horizon_days:
    h_seconds = h * 86400
    w_h = (
        Window.partitionBy("properties_deviceId")
        .orderBy(F.col("timeStamp").cast("long"))
        .rangeBetween(0, h_seconds)
    )
    df_encoded = df_encoded.withColumn(
        f"failure_within_{h}d",
        F.max(F.col("payload_fault_state").cast("int")).over(w_h)
    )

# Assign multi-class label (most urgent/shortest horizon = highest class)
sorted_horizons = sorted(horizon_days)  # shortest first
n_classes = len(sorted_horizons) + 1    # +1 for the "safe" class 0

label_expr = F.when(
    F.col(f"failure_within_{sorted_horizons[0]}d") == 1,
    F.lit(len(sorted_horizons))  # most urgent = highest class number
)
for i, h in enumerate(sorted_horizons[1:], start=1):
    label_expr = label_expr.when(
        F.col(f"failure_within_{h}d") == 1,
        F.lit(len(sorted_horizons) - i)
    )
label_expr = label_expr.otherwise(F.lit(0))

df_encoded = df_encoded.withColumn("failure_class", label_expr)

# --- Lag features (n_lags = 20) ---
n_lags = 20
w = Window.partitionBy("properties_deviceId").orderBy("timeStamp")

for L in range(1, n_lags + 1):
    for col_name in sensor_cols:
        df_encoded = df_encoded.withColumn(f"{col_name}{L}", F.lag(col_name, L).over(w))

lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]

# --- Break lineage: materialize to reset plan before adding more features ---
spark.sql("DROP TABLE IF EXISTS teams.sensorx._temp_lag_prep")
df_encoded.write.format("delta").mode("overwrite").saveAsTable("teams.sensorx._temp_lag_prep")
df_encoded = spark.table("teams.sensorx._temp_lag_prep")
print("Lineage broken after lags")

# --- Deviation features: 24-hour time-based rolling average ---
w_daily = (
    Window.partitionBy("properties_deviceId")
    .orderBy(F.col("timeStamp").cast("long"))
    .rangeBetween(-86400, 0)
)

dev_sensors = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
]
dev_cols = []

for col_name in dev_sensors:
    avg_col = f"{col_name}_avg_daily"
    dev_col = f"{col_name}_dev_daily"
    df_encoded = df_encoded.withColumn(avg_col, F.avg(col_name).over(w_daily))
    df_encoded = df_encoded.withColumn(dev_col, F.col(col_name) - F.col(avg_col))
    dev_cols.extend([avg_col, dev_col])

print(f"Added {len(dev_cols)} deviation features (24-hour window)")

# --- Drop nulls and assemble ---
df_clean = df_encoded.na.drop(subset=lag_cols + dev_cols)

feature_input_cols = sensor_cols + lag_cols + dev_cols + ["deviceId_index"]
assembler = VectorAssembler(inputCols=feature_input_cols, outputCol="features")
df_features = assembler.transform(df_clean)

df_features = df_features.select("timeStamp", "features", F.col("failure_class").alias("label"))

print(f"\nFeature vector size: {df_features.select('features').first()[0].size}")
print(f"  - {len(sensor_cols)} sensor cols")
print(f"  - {len(lag_cols)} lag cols")
print(f"  - {len(dev_cols)} deviation cols (24-hour window)")
print(f"  - 1 deviceId index")

# Chronological test train split 80/20
cutoff_epoch = df_features.selectExpr(
    "percentile_approx(CAST(timeStamp AS BIGINT), 0.8)"
).first()[0]

train = df_features.filter(F.col("timeStamp").cast("bigint") <= cutoff_epoch).cache()
test  = df_features.filter(F.col("timeStamp").cast("bigint") >  cutoff_epoch).cache()

In [0]:
train.write.mode("overwrite").saveAsTable("teams.sensorx.train_data")
test.write.mode("overwrite").saveAsTable("teams.sensorx.test_data")

In [0]:
train=spark.read.table("teams.sensorx.train_data")
test=spark.read.table("teams.sensorx.test_data")

In [0]:
# Class weighting (give minority class higher weight)
label_counts = train.groupBy("label").count().collect()
total = sum(row["count"] for row in label_counts)
weight_map = {row["label"]: total / (2.0 * row["count"]) for row in label_counts}

for lbl in sorted(weight_map):
    cnt = [r["count"] for r in label_counts if r["label"] == lbl][0]
    pct = cnt / total * 100
    print(f"  Label {lbl}: {cnt:,} samples ({pct:.1f}%) -> weight = {weight_map[lbl]:.4f}")
print(f"  Total: {total:,} samples")

# Add weight column to train and test
weight_expr = F.when(F.col("label") == 1, weight_map[1]).otherwise(weight_map[0])
train_w = train.withColumn("weight", weight_expr)
test_w = test.withColumn("weight", weight_expr)

In [0]:

# Raw fault_state distribution
print("Raw payload_fault_state distribution:")
df.groupBy("payload_fault_state").count().orderBy("payload_fault_state").show()

# After failure horizon
print(f"Failure horizon (H={H_days} days) distribution:")
df_clean.groupBy("failure_horizon").count().orderBy("failure_horizon").show()

Gradient Boosted trees with lag and horizon:

skoða weights, see feature importance for the trees


In [0]:
# GBT training (with OHE device ID features)
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    maxIter=30,
    seed=42
)

GBTmodel = gbt.fit(train_w)
gbtpredictions = GBTmodel.transform(test_w)

# --- Training accuracy ---
gbt_train_preds = GBTmodel.transform(train_w)
GBT_train_auc = auc_eval.evaluate(gbt_train_preds)

# --- Test evaluation ---
GBT_auc = auc_eval.evaluate(gbtpredictions)

# Recall (per class)
GBT_recall_0 = recall_eval.evaluate(gbtpredictions, {recall_eval.metricLabel: 0.0})
GBT_recall_1 = recall_eval.evaluate(gbtpredictions, {recall_eval.metricLabel: 1.0})

# F1 score (per class)
GBT_f1_0 = f1_eval.evaluate(gbtpredictions, {f1_eval.metricLabel: 0.0})
GBT_f1_1 = f1_eval.evaluate(gbtpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nGBT with {n_lags} lags + OHE deviceId (class-weighted)")
print(f"Training AUC: {GBT_train_auc:.4f}")
print(f"Test AUC:     {GBT_auc:.4f}")
print(f"Recall (label=0): {GBT_recall_0:.4f}")
print(f"Recall (label=1): {GBT_recall_1:.4f}")
print(f"F1     (label=0): {GBT_f1_0:.4f}")
print(f"F1     (label=1): {GBT_f1_1:.4f}")

# Confusion matrix
gbtpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

# --- Feature importance ---
ohe_size = GBTmodel.numFeatures - len(sensor_cols) - len(lag_cols)
ohe_names = [f"deviceId_ohe_{i}" for i in range(ohe_size)]
all_feature_names = sensor_cols + lag_cols + ohe_names

importances = GBTmodel.featureImportances.toArray().tolist()
feat_imp_df = spark.createDataFrame(
    zip(all_feature_names, importances), ["feature", "importance"]
).orderBy(F.desc("importance"))

print(f"\nTotal features: {GBTmodel.numFeatures} ({len(sensor_cols)} sensor + {len(lag_cols)} lag + {ohe_size} OHE)")
print(f"Top 20 features by importance:")
display(feat_imp_df.limit(20))

In [0]:
# --- RandomForest with class weighting + OHE deviceId ---
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    numTrees=100,
    seed=42
)
RFmodel = rf.fit(train_w)
RFpredictions = RFmodel.transform(test_w)

RFtrainingSummary = RFmodel.summary
RF_train_auc = auc_eval.evaluate(RFmodel.transform(train_w))
RF_auc = auc_eval.evaluate(RFpredictions)

# Recall (per class)
RF_recall_0 = recall_eval.evaluate(RFpredictions, {recall_eval.metricLabel: 0.0})
RF_recall_1 = recall_eval.evaluate(RFpredictions, {recall_eval.metricLabel: 1.0})

# F1 score (per class)
RF_f1_0 = f1_eval.evaluate(RFpredictions, {f1_eval.metricLabel: 0.0})
RF_f1_1 = f1_eval.evaluate(RFpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nRandom Forest with {n_lags} lags + OHE deviceId (class-weighted)")
print(f"Training AUC: {RF_train_auc:.4f}")
print(f"Test AUC:     {RF_auc:.4f}")
print(f"Recall (label=0): {RF_recall_0:.4f}")
print(f"Recall (label=1): {RF_recall_1:.4f}")
print(f"F1     (label=0): {RF_f1_0:.4f}")
print(f"F1     (label=1): {RF_f1_1:.4f}")

# Confusion matrix
RFpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

# --- Feature importance ---
ohe_size = RFmodel.numFeatures - len(sensor_cols) - len(lag_cols)
ohe_names = [f"deviceId_ohe_{i}" for i in range(ohe_size)]
all_feature_names = sensor_cols + lag_cols + ohe_names

rf_importances = RFmodel.featureImportances.toArray().tolist()
rf_feat_imp_df = spark.createDataFrame(
    zip(all_feature_names, rf_importances), ["feature", "importance"]
).orderBy(F.desc("importance"))

print(f"\nTotal features: {RFmodel.numFeatures} ({len(sensor_cols)} sensor + {len(lag_cols)} lag + {ohe_size} OHE)")
print(f"Top 20 features by importance:")
display(rf_feat_imp_df.limit(20))

In [0]:
# --- Balance ALL classes equally for MLP (train only) ---
label_counts = {row["label"]: row["count"] for row in train.groupBy("label").count().collect()}

print("Original class distribution:")
for lbl in sorted(label_counts):
    print(f"  Class {lbl}: {label_counts[lbl]:,}")

# Target count = largest minority class (excludes class 0)
minority_counts = {k: v for k, v in label_counts.items() if k != 0}
target_count = max(minority_counts.values())
print(f"\nTarget count per class: {target_count:,} (matching largest minority class)")

# Resample each class to target_count
balanced_parts = []
for lbl in sorted(label_counts):
    class_df = train.filter(F.col("label") == lbl)
    current_count = label_counts[lbl]

    if current_count > target_count:
        # Downsample (class 0 and any larger class)
        fraction = target_count / current_count
        resampled = class_df.sample(fraction=fraction, seed=42)
    elif current_count < target_count:
        # Oversample with replacement
        fraction = target_count / current_count
        resampled = class_df.sample(withReplacement=True, fraction=fraction, seed=42)
    else:
        resampled = class_df

    balanced_parts.append(resampled)

train_ds = balanced_parts[0]
for part in balanced_parts[1:]:
    train_ds = train_ds.unionAll(part)

print(f"\nBalanced training set:")
for row in train_ds.groupBy("label").count().orderBy("label").collect():
    print(f"  Class {row['label']}: {row['count']:,}")
print(f"  Total: {train_ds.count():,}")

# --- Normalize features for MLP ---
# Fit scaler on FULL train (preserves accurate mean/std statistics)
scaler = StandardScaler(
    inputCol="features",
    outputCol="features_scaled",
    withMean=True,
    withStd=True
)
scaler_model = scaler.fit(train)

# Transform balanced train and full test
train_norm = scaler_model.transform(train_ds).drop("features").withColumnRenamed("features_scaled", "features")
test_norm = scaler_model.transform(test).drop("features").withColumnRenamed("features_scaled", "features")

print(f"\nNormalized feature vector size: {train_norm.select('features').first()[0].size}")

Neural Network testing

-- check tuning learning rate, where to do that ? (mögulega step size) 

In [0]:
# Input layer = 133 features, output = 4 classes (multi-class horizon)
layers = [133, 16, 16, n_classes]

trainer = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    layers=layers,
    blockSize=128,
    seed=42,
    stepSize=0.001
)

MLPmodel = trainer.fit(train_norm)
MLPpredictions = MLPmodel.transform(test_norm)

# --- Evaluate per class (use MulticlassClassificationEvaluator for 4-class output) ---
accuracy_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
weighted_f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedFMeasure"
)

MLP_train_acc = accuracy_eval.evaluate(MLPmodel.transform(train_norm))
MLP_test_acc = accuracy_eval.evaluate(MLPpredictions)
MLP_weighted_f1 = weighted_f1_eval.evaluate(MLPpredictions)

print(f"\nMLP Neural Network ({n_classes}-class horizon, {n_lags} lags + deviation features)")
print(f"Layers: {layers}")
print(f"Training Accuracy: {MLP_train_acc:.4f}")
print(f"Test Accuracy:     {MLP_test_acc:.4f}")
print(f"Weighted F1:       {MLP_weighted_f1:.4f}")

for cls in range(n_classes):
    r = recall_eval.evaluate(MLPpredictions, {recall_eval.metricLabel: float(cls)})
    f = f1_eval.evaluate(MLPpredictions, {f1_eval.metricLabel: float(cls)})
    print(f"  Class {cls}: Recall={r:.4f}  F1={f:.4f}")

# Confusion matrix
print("\nConfusion matrix:")
MLPpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

In [0]:
# === MLP Comparison: With vs Without Downsampling ===

# Normalize full (imbalanced) train using same scaler from cell 11
train_norm_full = scaler_model.transform(train).drop("features").withColumnRenamed("features_scaled", "features")

# Train MLP on full imbalanced data (same hyperparameters)
print("Training MLP on FULL (imbalanced) data...")
trainer_full = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    layers=layers,
    blockSize=128,
    seed=42,
    stepSize=0.001
)

MLPmodel_full = trainer_full.fit(train_norm_full)
MLPpreds_full = MLPmodel_full.transform(test_norm)
print("Done. Comparing metrics...\n")

# --- Side-by-side comparison ---
ds_acc = accuracy_eval.evaluate(MLPpredictions)
ds_f1 = weighted_f1_eval.evaluate(MLPpredictions)
full_acc = accuracy_eval.evaluate(MLPpreds_full)
full_f1 = weighted_f1_eval.evaluate(MLPpreds_full)

print("=" * 85)
print("  MLP PERFORMANCE: DOWNSAMPLING COMPARISON")
print("=" * 85)
print(f"\n{'Metric':<25} {'No Downsampling':<20} {'With Downsampling':<20} {'Difference':<15}")
print("-" * 85)
print(f"{'Test Accuracy':<25} {full_acc:<20.4f} {ds_acc:<20.4f} {ds_acc - full_acc:<+15.4f}")
print(f"{'Weighted F1':<25} {full_f1:<20.4f} {ds_f1:<20.4f} {ds_f1 - full_f1:<+15.4f}")

print(f"\n{'Per-class Recall:':<25}")
for cls in range(n_classes):
    r_full = recall_eval.evaluate(MLPpreds_full, {recall_eval.metricLabel: float(cls)})
    r_ds = recall_eval.evaluate(MLPpredictions, {recall_eval.metricLabel: float(cls)})
    suffix = " (safe)" if cls == 0 else (" (imminent)" if cls == n_classes - 1 else "")
    print(f"  Class {cls}{suffix:<18} {r_full:<20.4f} {r_ds:<20.4f} {r_ds - r_full:<+15.4f}")

print(f"\n{'Per-class F1:':<25}")
for cls in range(n_classes):
    f_full = f1_eval.evaluate(MLPpreds_full, {f1_eval.metricLabel: float(cls)})
    f_ds = f1_eval.evaluate(MLPpredictions, {f1_eval.metricLabel: float(cls)})
    suffix = " (safe)" if cls == 0 else (" (imminent)" if cls == n_classes - 1 else "")
    print(f"  Class {cls}{suffix:<18} {f_full:<20.4f} {f_ds:<20.4f} {f_ds - f_full:<+15.4f}")

print("\n" + "=" * 85)
print("\n--- Confusion Matrix: NO downsampling ---")
MLPpreds_full.groupBy("label").pivot("prediction").count().orderBy("label").show()

print("--- Confusion Matrix: WITH downsampling ---")
MLPpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

In [0]:
import os
import mlflow
from mlflow.models import infer_signature
from pyspark.ml.functions import vector_to_array

# Required for Spark ML model logging on shared/serverless clusters
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

mlflow.set_registry_uri("databricks-uc")

CATALOG = "teams"
SCHEMA = "sensorx"
MLP_MODEL_NAME = f"{CATALOG}.{SCHEMA}.mlp_fault_prediction"

with mlflow.start_run(run_name="MLP_fault_prediction") as run:
    # Log parameters
    mlflow.log_param("model_type", "MultilayerPerceptronClassifier")
    mlflow.log_param("layers", str(layers))
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("stepSize", 0.001)
    mlflow.log_param("blockSize", 128)
    mlflow.log_param("n_lags", n_lags)
    mlflow.log_param("H_days", H_days)
    mlflow.log_param("normalized", True)

    # Log metrics
    mlflow.log_metric("AUC", MLP_auc)
    mlflow.log_metric("Training_AUC", MLP_training_auc)
    mlflow.log_metric("Recall_label_0", MLP_recall_0)
    mlflow.log_metric("Recall_label_1", MLP_recall_1)
    mlflow.log_metric("F1_label_0", MLP_f1_0)
    mlflow.log_metric("F1_label_1", MLP_f1_1)

    # Infer signature from a small sample
    sample_input = test_norm.select("features").limit(5)
    sample_output = MLPmodel.transform(sample_input).select("prediction")
    signature = infer_signature(
        sample_input.toPandas(),
        sample_output.toPandas()
    )

    # Convert DenseVector to array for JSON-serializable input_example
    input_example = (
        sample_input.limit(1)
        .select(vector_to_array("features").alias("features"))
        .toPandas()
    )

    # Log and register the Spark ML model to Unity Catalog
    model_info = mlflow.spark.log_model(
        MLPmodel,
        artifact_path="mlp_model",
        signature=signature,
        input_example=input_example,
        registered_model_name=MLP_MODEL_NAME,
    )

    print(f"Model registered to: {MLP_MODEL_NAME}")
    print(f"Run ID: {run.info.run_id}")
    print(f"Model URI: {model_info.model_uri}")
    print(f"AUC: {MLP_auc:.4f}")
    print(f"Recall (label=1): {MLP_recall_1:.4f}")
    print(f"F1 (label=1): {MLP_f1_1:.4f}")

In [0]:
# --- Model Comparison Table ---
comparison_data = [
    ("Random Forest", RF_train_auc, RF_auc, RF_recall_0, RF_recall_1, RF_f1_0, RF_f1_1),
    ("Gradient Boosted Trees", GBT_train_auc, GBT_auc, GBT_recall_0, GBT_recall_1, GBT_f1_0, GBT_f1_1),
    ("MLP Neural Network", MLP_training_auc, MLP_auc, MLP_recall_0, MLP_recall_1, MLP_f1_0, MLP_f1_1),
]

comparison_df = spark.createDataFrame(
    comparison_data,
    ["Model", "Training_AUC", "Test_AUC", "Recall_0", "Recall_1", "F1_0", "F1_1"]
)

print("Model Comparison (15-day fault prediction horizon, 20 lags):")
display(comparison_df)

In [0]:
# Run MLP predictions on the full test set and save as a Delta table
mlp_all_preds = MLPmodel.transform(test_norm)

# Join back with original data to get deviceId and actual fault_state
original_cols = df_encoded.select(
    "timeStamp", "properties_deviceId",
    F.col("payload_fault_state").cast("int").alias("actual_fault_state")
)

mlp_timeline = (
    mlp_all_preds
    .join(original_cols, on="timeStamp", how="inner")
    .select(
        "timeStamp",
        "properties_deviceId",
        F.col("actual_fault_state").alias("fault_now"),
        F.col("label").alias("fault_within_15d"),
        F.col("prediction").cast("int").alias("predicted_fault_within_15d"),
        vector_to_array("probability")[1].alias("prob_fault")
    )
    .orderBy("properties_deviceId", "timeStamp")
)

# Save to Delta
spark.sql("DROP TABLE IF EXISTS teams.sensorx.mlp_predictions_delta")
mlp_timeline.write.saveAsTable("teams.sensorx.mlp_predictions_delta")

print(f"Predictions saved to teams.sensorx.mlp_predictions_delta")
print(f"Total predictions: {mlp_timeline.count()}")
display(mlp_timeline.limit(20))

find version for the thing below in spark


In [0]:
%pip install shap

In [0]:
import os
import mlflow
from pyspark.ml.feature import StandardScaler

# --- Load model from MLflow (version 3: 161 features with OHE device IDs) ---
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"
MLPmodel = mlflow.spark.load_model("models:/teams.sensorx.mlp_fault_prediction/3")

# --- Read saved train/test and normalize ---
train = spark.read.table("teams.sensorx.train_features_delta")
test = spark.read.table("teams.sensorx.test_features_delta")

scaler = StandardScaler(inputCol="features", outputCol="features_scaled", withMean=True, withStd=True)
scaler_model = scaler.fit(train)
train_norm = scaler_model.transform(train).drop("features").withColumnRenamed("features_scaled", "features")
test_norm = scaler_model.transform(test).drop("features").withColumnRenamed("features_scaled", "features")

# --- Reconstruct feature names (sensor + lag + OHE) ---
exclude = {"properties_deviceId", "timeStamp", "payload_fault_state", "serialNumber", "payload_fault_faultName", "GeneratorType"}
original_cols = spark.read.table("teams.sensorx.df_sx_800_with_delta").columns
sensor_cols = [c for c in original_cols if c not in exclude]
n_lags = 20
lag_cols = [f"{col}{L}" for L in range(1, n_lags + 1) for col in sensor_cols]

# OHE feature count = total features - sensor - lag
total_features = train_norm.select("features").first()[0].size
ohe_size = total_features - len(sensor_cols) - len(lag_cols)
ohe_names = [f"deviceId_ohe_{i}" for i in range(ohe_size)]
feature_names = sensor_cols + lag_cols + ohe_names

print(f"Model loaded. Features: {len(feature_names)} ({len(sensor_cols)} sensor + {len(lag_cols)} lag + {ohe_size} OHE)")

In [0]:
import numpy as np
import pandas as pd
import shap
from pyspark.sql import Row
from pyspark.ml.linalg import Vectors

# Background data (mixed classes)
bg_pdf = train_norm.select("features").limit(100).toPandas()
X_bg = np.vstack([r.toArray() for r in bg_pdf["features"]])

# Explanation data: fault-positive samples only (label == 1)
fault_pdf = test_norm.filter("label = 1").select("features").limit(10).toPandas()
X_fault = np.vstack([r.toArray() for r in fault_pdf["features"]])
print(f"Fault-positive explanation samples: {len(X_fault)}")

def predict_proba(X):
    if isinstance(X, pd.DataFrame):
        X = X.values
    rows = [Row(features=Vectors.dense(r.tolist())) for r in X]
    results = MLPmodel.transform(spark.createDataFrame(rows)).select("probability").collect()
    return np.array([r["probability"].toArray() for r in results])

# Check what the model actually predicts for these fault samples
fault_preds = predict_proba(X_fault)
print(f"Model predictions for fault samples (prob_no_fault, prob_fault):")
for i, p in enumerate(fault_preds):
    print(f"  Sample {i}: no_fault={p[0]:.4f}, fault={p[1]:.4f} -> {'FAULT' if p[1] > 0.5 else 'NO FAULT'}")

explainer = shap.KernelExplainer(predict_proba, shap.sample(pd.DataFrame(X_bg, columns=feature_names), 50))
print(f"SHAP base value (expected output): {explainer.expected_value}")

X_fault_df = pd.DataFrame(X_fault, columns=feature_names)
shap_values = explainer.shap_values(X_fault_df, nsamples=200)

sv = np.array(shap_values)
print(f"SHAP output shape: {sv.shape}")
if sv.ndim == 3:
    sv = sv[:, :, 1]
elif isinstance(shap_values, list):
    sv = np.array(shap_values[1])

print(f"Class 1 SHAP range: [{sv.min():.4f}, {sv.max():.4f}]")
shap.summary_plot(sv, X_fault_df)

In [0]:
import numpy as np
import pandas as pd
import shap
from pyspark.sql import Row
from pyspark.ml.linalg import Vectors

# Background data (mixed classes)
bg_pdf = train_norm.select("features").limit(100).toPandas()
X_bg = np.vstack([r.toArray() for r in bg_pdf["features"]])

# Explanation data: mixed labels (overall importance)
all_pdf = test_norm.select("features").limit(10).toPandas()
X_all = np.vstack([r.toArray() for r in all_pdf["features"]])
print(f"Overall explanation samples: {len(X_all)}")

def predict_proba(X):
    if isinstance(X, pd.DataFrame):
        X = X.values
    rows = [Row(features=Vectors.dense(r.tolist())) for r in X]
    results = MLPmodel.transform(spark.createDataFrame(rows)).select("probability").collect()
    return np.array([r["probability"].toArray() for r in results])

explainer = shap.KernelExplainer(predict_proba, shap.sample(pd.DataFrame(X_bg, columns=feature_names), 50))

X_all_df = pd.DataFrame(X_all, columns=feature_names)
shap_values_all = explainer.shap_values(X_all_df, nsamples=200)

sv_all = np.array(shap_values_all)
if sv_all.ndim == 3:
    sv_all = sv_all[:, :, 1]
elif isinstance(shap_values_all, list):
    sv_all = np.array(shap_values_all[1])

shap.summary_plot(sv_all, X_all_df)